In [35]:
from typing import Dict, List, Tuple, Optional
from procompa.combfold_eval.config import Config
from procompa.combfold_eval.sequences import UniProtSequences
from procompa.combfold_eval.pipeline import _row_status, locate_models
import glob
# from .compare import compare_one

def _process_row(idx: int, row: "pd.Series", cfg: Config, seqs: UniProtSequences,
                 json_dir: str, save_json: bool) -> Tuple[int, List[Dict], List[Dict], List[Dict], List[str]]:
    """Score one mapping-file row (one complex_ac). Fully self-contained: every
    resource it touches (compare_one's temp work dir, per-complex JSON path,
    reference/sequence caches guarded internally) is either unique to this call
    or lock-protected, so this function is safe to run concurrently for
    different rows. Returns (idx, summary_rows, chain_rows, iface_rows, log_lines)
    so the caller can put results back in original row order regardless of
    which order the workers finish in.
    """
    summary_rows: List[Dict] = []
    chain_rows: List[Dict] = []
    iface_rows: List[Dict] = []
    log: List[str] = []

    cac = str(row["complex_ac"]).strip()
    pdb = str(row["pdb_id"]).strip()
    # Carry manifest-driven provenance through to non-scored status rows too.
    _ssrc = str(row.get("stoich_source", "")) if "stoich_source" in row else ""
    _ftype = str(row.get("cf_folder_type", "")) if "cf_folder_type" in row else ""
    while True:
        # ---- locate model(s) ----
        complex_dir = None
        models: List[str] = []
        model_col = row.get("model") if "model" in row else None
        if model_col and str(model_col) != "nan":
            print("we are  in 1")
            mc = str(model_col)
            models = sorted(glob.glob(mc)) if any(c in mc for c in "*?[") else (
                [mc] if os.path.exists(mc) else [])
        else:
            print("we are in 2")
            print("Inputs:\n combfold_base: ", cfg.combfold_base, "\n cac: ", cac, "\n row.get(folder): ", row.get("folder") if "folder" in row else None, "\n cf_model_globs: ", cfg.cf_model_globs)
            complex_dir, models = locate_models(
                cfg.combfold_base, cac,
                row.get("folder") if "folder" in row else None,
                cfg.cf_model_globs)
            print(f"locate_models returned: {complex_dir}, {models}")
            if not models:
                for g in (f"{cac.replace('-', '_')}*.pdb", f"{cac}*.pdb"):
                    hits = sorted(glob.glob(os.path.join(cfg.combfold_base, g)))
                    if hits:
                        models = hits
                        break
        print(models)
        if not models:
            summary_rows.append(_row_status(cac, pdb, "no_model_found",
                                            stoich_source=_ssrc, cf_folder_type=_ftype))
            log.append(f"{cac}: no CombFold model found")
            return idx, summary_rows, chain_rows, iface_rows, log

        # # ---- confidence ----
        # conf: Dict[int, Tuple[float, int]] = {}
        # if complex_dir:
        #     conf = parse_confidence(os.path.join(complex_dir, "assembled_results",
        #                                          cfg.cf_confidence_name))
        #     if not conf:
        #         conf = parse_confidence(os.path.join(complex_dir, cfg.cf_confidence_name))

        # # ---- candidate UniProts ----
        # candidates, csource, stoich = resolve_candidates(row, complex_dir, models[0], seqs)
        # if not candidates:
        #     summary_rows.append(_row_status(cac, pdb, "no_uniprots_resolved",
        #                                     stoich_source=_ssrc, cf_folder_type=_ftype))
        #     log.append(f"{cac}: could not resolve candidate UniProts")
        #     return idx, summary_rows, chain_rows, iface_rows, log

        # # ---- local reference override ----
        # local_ref = None
        # if "local_ref" in row and str(row.get("local_ref")) != "nan" and row.get("local_ref"):
        #     lr = str(row["local_ref"])
        #     local_ref = lr if os.path.exists(lr) else None

        # model_list = models if cfg.score_all_clusters else models[:1]
        # for i, mp in enumerate(model_list):
        #     cidx = cluster_idx_from_filename(mp)
        #     if cidx < 0:
        #         cidx = i
        #     score, rank = conf.get(cidx, (float("nan"), cidx))
        #     meta = {"complex_ac": cac, "pdb_id": pdb, "cf_cluster": cidx,
        #             "cf_confidence": score, "cf_rank": rank,
        #             "candidate_source": csource, "expected_stoich": stoich,
        #             "stoich_source": str(row.get("stoich_source", "")) if "stoich_source" in row else "",
        #             "cf_folder_type": str(row.get("cf_folder_type", "")) if "cf_folder_type" in row else ""}
        #     res = compare_one(mp, pdb, candidates, cfg, meta=meta, local_au=local_ref)
        #     for s in res["summary"]:
        #         s.setdefault("candidate_source", csource)
        #     summary_rows.extend(res["summary"])
        #     chain_rows.extend(res["per_chain"])
        #     iface_rows.extend(res["per_interface"])
        #     if save_json:
        #         # Include stoich_source + folder type in the JSON filename to
        #         # avoid collisions when one complex_ac has multiple scored
        #         # folders (manifest-driven expansion).
        #         ftype = meta["cf_folder_type"]
        #         ssrc = meta["stoich_source"].replace("+", "_")
        #         suffix = f"_{ftype}_{ssrc}" if (ftype or ssrc) else ""
        #         with open(os.path.join(json_dir, f"{cac}_c{cidx}{suffix}.json"), "w") as fh:
        #             json.dump(res["json"], fh, indent=2, default=str)
        #     log.append(f"{cac} cluster {cidx}: {len(res['summary'])} form(s) scored "
        #                f"(candidates from {csource})")
        break
    # except Exception as e:  # never abort the whole batch
    #     summary_rows.append(_row_status(cac, pdb, "error", flags=str(e)[:200],
    #                                     stoich_source=_ssrc, cf_folder_type=_ftype))
    #     log.append(f"{cac}: ERROR {type(e).__name__}: {e}")
    #     print(log)

In [10]:
tmpl_path = "/cluster/project/beltrao/kdammer/master_thesis/src/procompa/combfold_eval/tmp"

import os 
import pickle
with open(os.path.join(tmpl_path, "cfg.pkl"), "rb") as fh:
    cfg: Config = pickle.load(fh)
with open(os.path.join(tmpl_path, "selected_rows.pkl"), "rb") as fh:
    selected_rows = pickle.load(fh)
# with open(os.path.join(tmpl_path, "seqs.pkl"), "rb") as fh:
#     seqs: UniProtSequences = pickle.load(fh)

In [11]:
uniprot_csv_path = "/cluster/project/beltrao/kdammer/master_thesis/data/iPTM_and_pLDDT/all_CF_YM_yeast_proteins_uniprot_mapped_sequences.csv"
seqs = UniProtSequences(uniprot_csv_path, cache_dir=os.path.join(cfg.ref_cache, "_uniprot"))

In [12]:
seqs

In [34]:
selected_rows[0]

(0,
 complex_ac                 CPX-8172
 n_proteins                        2
 pdb_id                         2Z5B
 match_class         exact_pdb_match
 stoich_source     no_manifest_entry
 cf_folder_type                     
 Name: 5, dtype: object)

In [31]:
idx, row = selected_rows[0]
_process_row(idx, row, cfg, seqs, json_dir=tmpl_path, save_json=True)

we are in 2
Inputs:
 combfold_base:  /cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t6_CF_test_Example_for_RM_TM/CombFold 
 cac:  CPX-8172 
 row.get(folder):  None 
 cf_model_globs:  ('assembled_results/output_clustered_*.pdb', '*_output.pdb', 'output_clustered_*.pdb', '*.pdb')
locate_models returned: None, []
[]


(0,
 [{'complex_ac': 'CPX-8172',
   'pdb_id': '2Z5B',
   'cf_cluster': '',
   'cf_rank': '',
   'cf_confidence': '',
   'ref_form': '',
   'ref_assembly_id': '',
   'is_primary_ref': '',
   'n_shared_subunits': '',
   'missing_uniprots': '',
   'extra_uniprots': '',
   'complex_TM_ref': '',
   'complex_TM_mod': '',
   'global_rmsd_nocyc': '',
   'global_rmsd_wcyc': '',
   'n_res_global': '',
   'coverage_global': '',
   'mean_dockq': '',
   'pairing_source': '',
   'candidate_source': '',
   'ref_select_reason': '',
   'stoich_source': 'no_manifest_entry',
   'cf_folder_type': '',
   'status': 'no_model_found',
   'flags': ''}],
 [],
 [],
 ['CPX-8172: no CombFold model found'])

In [ ]:
complexes = None # i think this is set by "only"  in main of 02_combfold_eval.py 

selected: List[Tuple[int, "pd.Series"]] = []
manifest_map: Optional[Dict[str, "pd.Series"]] = None
if cfg.manifest_csv:
    manifest_map = _manifest.load_manifest(cfg.manifest_csv)

key = 0
for _idx, row in df.iterrows():
    cac = str(row["complex_ac"]).strip()
    if complexes and cac not in complexes:
        continue

    if manifest_map is not None:
        mrow = manifest_map.get(cac)
        if mrow is None:
            row = row.copy()
            row["stoich_source"] = "no_manifest_entry"
            row["cf_folder_type"] = ""
            selected.append((key, row))
            key += 1
            continue
        expanded = False
        for label, stoich in _manifest.candidate_stoichiometries(mrow):
            for suffix, folder in _manifest.resolve_folders(
                    cfg.combfold_base, stoich, cfg.cf_folder_suffixes):
                r = row.copy()
                r["folder"] = folder
                r["stoich_source"] = label
                r["cf_folder_type"] = suffix
                selected.append((key, r))
                key += 1
                expanded = True
        if not expanded:
            row = row.copy()
            row["stoich_source"] = "manifest_entry_but_no_folder"
            row["cf_folder_type"] = ""
            selected.append((key, row))
            key += 1
    else:
        selected.append((key, row))
        key += 1